In [ ]:
import csv
import random
import os

# Configuração
filename = "work/dados_gigantes.csv"
num_linhas = 10_000_000  # 10 Milhões de linhas (Ajuste conforme a RAM do seu PC)

print(f"Gerando {num_linhas} linhas de dados... aguarde.")

with open(filename, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["id", "categoria", "valor", "status"]) # Header
    
    # Gerando dados dummy
    # Vamos simular transações financeiras
    for i in range(num_linhas):
        writer.writerow([
            i, 
            random.choice(['A', 'B', 'C', 'D', 'E']), 
            random.uniform(10.0, 1000.0),
            random.choice(['Aprovado', 'Pendente', 'Cancelado'])
        ])

print(f"Arquivo {filename} gerado com sucesso! Tamanho aprox: {os.path.getsize(filename) / (1024*1024):.2f} MB")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

# 1. Inicializa o Spark
spark = SparkSession.builder \
    .appName("DemoAula1_Cache") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Ajuste o caminho se necessário (ex: "work/dados_gigantes.csv")
filename = "work/dados_gigantes.csv"

# 2. Leitura
print("--- Lendo arquivo (Lazy)... ---")
df = spark.read.csv(filename, header=True, inferSchema=True)

# 3. O PULO DO GATO: Marcamos o DataFrame para ser guardado na RAM
# Nota: O cache só acontece de verdade quando uma AÇÃO é chamada.
df.cache()

# --- EXECUÇÃO 1 (Lenta: Disco -> CPU -> RAM) ---
print("\n--- Execução 1: Processando e enchendo o Cache ---")
start_time_1 = time.time()

# Ação forçada para garantir que o Spark leia tudo e guarde na memória
# O count() obriga o Spark a passar por todas as linhas.
qtd = df.count()

end_time_1 = time.time()
tempo_1 = end_time_1 - start_time_1
print(f"Resultado: {qtd} linhas")
print(f"Tempo (Disco + Cache): {tempo_1:.4f} segundos")


# --- EXECUÇÃO 2 (Rápida: RAM -> CPU) ---
print("\n--- Execução 2: Usando dados da Memória ---")
start_time_2 = time.time()

# Agora fazemos a agregação pesada. Como os dados já estão na RAM (do passo anterior),
# ele não lê do disco.
resultado = df.groupBy("categoria").agg(F.sum("valor").alias("total"))
resultado.show()

end_time_2 = time.time()
tempo_2 = end_time_2 - start_time_2
print(f"Tempo (Memória): {tempo_2:.4f} segundos")

# Comparativo final
print(f"\n>>> O Spark com Cache foi {tempo_1 / tempo_2:.1f}x mais rápido na segunda vez.")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

# 1. Inicializa o Spark
spark = SparkSession.builder \
    .appName("DemoAula1_Cache") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Ajuste o caminho se necessário (ex: "work/dados_gigantes.csv")
filename = "work/dados_gigantes.csv"

# 2. Leitura
print("--- Lendo arquivo (Lazy)... ---")
df = spark.read.csv(filename, header=True, inferSchema=True)

# 3. O PULO DO GATO: Marcamos o DataFrame para ser guardado na RAM
# Nota: O cache só acontece de verdade quando uma AÇÃO é chamada.
df.cache()

# --- EXECUÇÃO 1 (Lenta: Disco -> CPU -> RAM) ---
print("\n--- Execução 1: Processando e enchendo o Cache ---")
start_time_1 = time.time()

# Ação forçada para garantir que o Spark leia tudo e guarde na memória
# O count() obriga o Spark a passar por todas as linhas.
qtd = df.count()

end_time_1 = time.time()
tempo_1 = end_time_1 - start_time_1
print(f"Resultado: {qtd} linhas")
print(f"Tempo (Disco + Cache): {tempo_1:.4f} segundos")


# --- EXECUÇÃO 2 (Rápida: RAM -> CPU) ---
print("\n--- Execução 2: Usando dados da Memória ---")
start_time_2 = time.time()

# Agora fazemos a agregação pesada. Como os dados já estão na RAM (do passo anterior),
# ele não lê do disco.
resultado = df.groupBy("categoria").agg(F.sum("valor").alias("total"))
resultado.show()

end_time_2 = time.time()
tempo_2 = end_time_2 - start_time_2
print(f"Tempo (Memória): {tempo_2:.4f} segundos")

# Comparativo final
print(f"\n>>> O Spark com Cache foi {tempo_1 / tempo_2:.1f}x mais rápido na segunda vez.")